In [ ]:
import os, cv2, numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm

In [ ]:
!pip install git+https://github.com/jacobgil/pytorch-grad-cam.git -q

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 25
NUM_CLASSES = 7

IMG_DIR1 = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1'
IMG_DIR2 = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2'
MASK_DIR = '/kaggle/input/datasets/tschandl/ham10000-lesion-segmentations/HAM10000_segmentations_lesion_tschandl'
CSV_PATH = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv'

In [ ]:
def get_image_path(image_id):
    p1 = os.path.join(IMG_DIR1, image_id + '.jpg')
    p2 = os.path.join(IMG_DIR2, image_id + '.jpg')
    
    if os.path.exists(p1):
        return p1
    elif os.path.exists(p2):
        return p2
    else:
        raise FileNotFoundError(image_id)

In [ ]:
df = pd.read_csv(CSV_PATH)

label_map = {l:i for i,l in enumerate(df['dx'].unique())}
df['label'] = df['dx'].map(label_map)

from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label'])

In [ ]:
train_tf = A.Compose([
    A.RandomResizedCrop(size=(224, 224), scale=(0.8, 1.0)),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=45),
    A.ColorJitter(),
    A.GaussianBlur(p=0.3),
    A.CoarseDropout(p=0.5),
    A.Normalize(),
    ToTensorV2()
])

In [ ]:
class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(3,64,3,1,1), nn.ReLU())
        self.enc2 = nn.Sequential(nn.Conv2d(64,128,3,1,1), nn.ReLU())
        self.pool = nn.MaxPool2d(2)

        self.up = nn.ConvTranspose2d(128,64,2,2)
        self.dec = nn.Sequential(nn.Conv2d(128,64,3,1,1), nn.ReLU())

        self.out = nn.Conv2d(64,1,1)

    def forward(self,x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        d = self.up(e2)
        d = self.dec(torch.cat([d,e1],1))
        return self.out(d)

In [ ]:
class SegDS(Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        id = self.df.iloc[idx]['image_id']

        img = cv2.imread(get_image_path(id))
        img = cv2.resize(img,(256,256))/255.0
        img = torch.tensor(img).permute(2,0,1).float()

        mask_path = os.path.join(MASK_DIR, id + '_segmentation.png')
        if os.path.exists(mask_path):
            mask = cv2.imread(mask_path,0)
        else:
            mask = np.zeros((256,256))

        mask = cv2.resize(mask,(256,256))
        mask = torch.tensor((mask>127).astype(float)).unsqueeze(0)

        return img, mask

In [ ]:
seg_model = UNet().to(DEVICE)
opt = torch.optim.Adam(seg_model.parameters(), 1e-4)
loss_fn = nn.BCEWithLogitsLoss()

loader = DataLoader(SegDS(train_df), batch_size=8, shuffle=True)

for e in range(10):
    seg_model.train()
    for x,y in loader:
        x,y = x.to(DEVICE), y.to(DEVICE)

        pred = seg_model(x)
        loss = loss_fn(pred,y)

        opt.zero_grad()
        loss.backward()
        opt.step()

    print("Seg Epoch", e)